# Conectar deployment job

El bundle crea el job. Este notebook sólo lo conecta al modelo registrado; en local genera un manifiesto del DAG.

In [ ]:
from iris_mlflow_utils import build_deployment_config, build_runtime_config, detect_runtime, get_runtime_parameter, write_manifest

runtime_mode = detect_runtime()
config = build_runtime_config(model_slug='random_forest')
deployment_config = build_deployment_config()
if runtime_mode == 'local':
    manifest_path = config.deployment_manifest_path.with_name('local_deployment_job_manifest.json')
    manifest = {
        'runtime': 'local',
        'status': 'simulated',
        'job_name': deployment_config.job_name,
        'tasks': ['evaluate_model', 'Approval_Check', 'deploy_model'],
        'model_name': deployment_config.model_name,
        'deployment_skipped': True,
    }
    write_manifest(manifest_path, manifest)
    print(manifest)
else:
    from mlflow.tracking import MlflowClient
    model_name = get_runtime_parameter('model_name', deployment_config.model_name, runtime_mode)
    job_id = get_runtime_parameter('deployment_job_id', '', runtime_mode)
    if not job_id.isdigit():
        raise ValueError('deployment_job_id debe ser el ID numérico inyectado por el bundle.')
    registry = MlflowClient(registry_uri='databricks-uc')
    registry.update_registered_model(name=model_name, deployment_job_id=job_id)
    print({'runtime': runtime_mode, 'job_id': job_id, 'action': 'connected', 'model_name': model_name, 'endpoint': deployment_config.endpoint_name})
